# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain 0.3.26 Update

This notebook has been updated to work with **LangChain 0.3.26**.

**Key changes from older versions:**
- ❌ Removed: `VectorstoreIndexCreator`, `RetrievalQA`, `QAGenerateChain`, `QAEvalChain`
- ✅ Added: Direct vectorstore creation, LCEL chains, Ragas integration
- ✅ All chains now use the LCEL pipe operator (`|`)

Run all cells sequentially to evaluate your LLM application!

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv('../cred.env')

# retrieve API keys from .env file

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

### Example 1

#### Create our QandA application

In [4]:
# LangChain 0.3.26 imports - updated
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.document_loaders import CSVLoader, TextLoader
from langchain.vectorstores import Chroma
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [3]:
file = './data/OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)
data = loader.load()

In [4]:
# !pip install --upgrade --force-reinstall sentence-transformers

In [5]:
# LangChain 0.3.26: Create vectorstore directly (VectorstoreIndexCreator removed)
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={'device': 'cpu'})
vector_db = Chroma.from_documents(data, embedding, persist_directory="./chroma_db")
retriever = vector_db.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9001.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# LangChain 0.3.26: Create retrieval chain using LCEL
llm = ChatOpenAI(temperature=0.0, model="gpt-4o-mini")

# Keep retrieval small to avoid token burst / 429 errors
retriever = vector_db.as_retriever(search_kwargs={"k": 2})

def format_docs(docs):
    max_chars_per_doc = 1200
    return "\n\n".join(doc.page_content[:max_chars_per_doc] for doc in docs)

# Define prompt template
qa_prompt = ChatPromptTemplate.from_template(
    """Answer the question based on the following context. Keep the answer concise.

{context}

Question: {question}
Answer:"""
)

# Create retrieval chain using LCEL
qa = (
    {"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | qa_prompt
    | llm
    | StrOutputParser()
)

#### Coming up with test datapoints

In [7]:
data[10]

Document(metadata={'source': './data/OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.")

In [8]:
data[11]

Document(metadata={'source': './data/OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

#### Hard-coded examples

In [9]:
from langchain.prompts import PromptTemplate

In [10]:
# LangChain 0.3.26: Use LCEL with PromptTemplate and StrOutputParser
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

# Define the prompt template
prompt_template = PromptTemplate(
    input_variables=["query"],
    template="""Examples:
1. Query: Do the Cozy Comfort Pullover Set have side pockets?
   Answer: Yes
2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?
   Answer: The DownTek collection
Query: {query}
Answer:"""
)

# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# Create chain using LCEL (0.3.26 style)
llm_chain = prompt_template | llm | parser

# Example query
query = "Is the Cozy Comfort Pullover Set available in different colors?"

# Run the chain
result = llm_chain.invoke({"query": query})

# Print the result
print(result)

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Yes, the Cozy Comfort Pullover Set is available in different colors.


#### LLM-Generated examples

In [11]:
# LangChain 0.3.26: QAGenerateChain removed - create manual Q&A generation
import json

qa_gen_template = ChatPromptTemplate.from_template(
    """Generate a question and answer pair based on this document:

{doc}

Respond with ONLY valid JSON (no markdown) with 'query' and 'answer' keys."""
)

qa_gen_chain = qa_gen_template | llm | StrOutputParser()

# Generate examples from documents
new_examples = []
for i, doc in enumerate(data[:5]):
    try:
        response = qa_gen_chain.invoke({"doc": doc.page_content[:300]})
        qa_pair = json.loads(response)
        new_examples.append(qa_pair)
        print(f"Generated Q&A {i+1}")
    except Exception as e:
        print(f"Skipped document {i}: {str(e)[:50]}")
        new_examples.append({
            "query": f"What product is in example {i}?",
            "answer": "Product information from catalog"
        })

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Generated Q&A 1


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Generated Q&A 2


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Generated Q&A 3


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Generated Q&A 4


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Generated Q&A 5


In [12]:
new_examples[0]

{'query': "What are the key features of the Women's Campside Oxfords?",
 'answer': "The Women's Campside Oxfords feature a super-soft canvas, thick cushioning, and quality construction that provides a broken-in feel from the first time you wear them."}

In [13]:
data[0]

Document(metadata={'source': './data/OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

In [14]:
# Flatten the examples for batch processing
d_flattened = new_examples
print(f"Generated {len(d_flattened)} examples")
d_flattened

Generated 5 examples


[{'query': "What are the key features of the Women's Campside Oxfords?",
  'answer': "The Women's Campside Oxfords feature a super-soft canvas, thick cushioning, and quality construction that provides a broken-in feel from the first time you wear them."},
 {'query': 'What are the dimensions of the small and medium sizes of the Recycled Waterhog Dog Mat?',
  'answer': 'The small size has dimensions of 18" x 28" and the medium size has dimensions of 22.5" x 34.5".'},
 {'query': "What features does the Infant and Toddler Girls' Coastal Chill Swimsuit have?",
  'answer': 'The swimsuit features bright colors, ruffles, exclusive whimsical prints, four-way-stretch and chlorine-resistant fabric that keeps its shape and resists snags, and is made from UPF 50+ rated fabric.'},
 {'query': 'What is the design style of the Refresh Swimwear V-Neck Tankini?',
  'answer': 'The Refresh Swimwear V-Neck Tankini features an eye-catching colorblock style.'},
 {'query': 'What technology is used in the EcoFl

#### Combine examples

In [15]:
# examples += new_example
examples += d_flattened

In [16]:
examples[0]

{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
 'answer': 'Yes'}

In [17]:
# Test the QA chain (0.3.26 returns just the answer string)
result = qa.invoke(examples[0]["query"])
print("Answer:", result)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Answer: Yes, the Cozy Cuddles Knit Pullover Set has side pockets.


### Manual Evaluation - Fun part

In [18]:
import langchain
langchain.debug = True

In [19]:
qa.invoke(examples[0]["query"])

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Do the Cozy Comfort Pullover Set have side pockets?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question>] Entering Chain run with input:
{
  "input": "Do the Cozy Comfort Pullover Set have side pockets?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Do the Cozy Comfort Pullover Set have side pockets?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnablePassthrough] Entering Chain run with input:
{
  "input": "Do the Cozy Comfort Pullover Set have side pockets?"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnablePassthrough] s] Exiting Chain run with output:
{
  "output": "Do the Cozy Comfort Pullover Set have side pockets?"
}
[chain/start] [chain:RunnableSequence > chain:Runn

'Yes, the Cozy Cuddles Knit Pullover Set has side pockets.'

In [20]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [21]:
examples += d_flattened

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [22]:
examples

[{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
  'answer': 'Yes'},
 {'query': 'What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection'},
 {'query': "What are the key features of the Women's Campside Oxfords?",
  'answer': "The Women's Campside Oxfords feature a super-soft canvas, thick cushioning, and quality construction that provides a broken-in feel from the first time you wear them."},
 {'query': 'What are the dimensions of the small and medium sizes of the Recycled Waterhog Dog Mat?',
  'answer': 'The small size has dimensions of 18" x 28" and the medium size has dimensions of 22.5" x 34.5".'},
 {'query': "What features does the Infant and Toddler Girls' Coastal Chill Swimsuit have?",
  'answer': 'The swimsuit features bright colors, ruffles, exclusive whimsical prints, four-way-stretch and chlorine-resistant fabric that keeps its shape and resists snags, and is made from UPF 50+ rated fabric.'},
 {'query': 

In [23]:
# Batch predictions (LCEL chain expects a list of strings, not dicts)
qa_inputs = [item["query"] if isinstance(item, dict) else str(item) for item in examples]
predictions = qa.batch(qa_inputs)
print(f"Generated {len(predictions)} predictions")

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Generated 12 predictions


In [24]:
predictions

['Yes, the Cozy Cuddles Knit Pullover Set has side pockets.',
 'The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.',
 "The key features of the Easy-Care Long-Sleeve Gingham Oxford Shirt include:\n\n- 100% cotton, wrinkle-resistant pinpoint fabric\n- Relaxed fit that falls at low hip\n- Finely tailored with a narrow placket and pearlized buttons\n- Taped seams for minimal ironing\n- Women's sizes include bust darts\n- Machine washable and dryable\n- Imported",
 'Small - 18" x 28"; Medium - 22.5" x 34.5".',
 "The Infant and Toddler Girls' Coastal Chill Swimsuit features bright colors, ruffles, exclusive prints, four-way-stretch and chlorine-resistant fabric, UPF 50+ sun protection, crossover no-slip straps, and a fully lined bottom for a secure fit. It is machine washable and should be line dried.",
 'The design style of the Renew Refresh Swimwear V-Neck Tankini is fitted with a flattering V-neck silhouette and lightweight racerback straps.',
 'The EcoFlex 3L 

In [42]:
# LangChain 0.3.26: QAEvalChain removed - create simple LLM-based evaluator
eval_template = ChatPromptTemplate.from_template(
    """Evaluate if the predicted answer is correct based on the reference answer.

Reference Answer: {reference_answer}
Predicted Answer: {predicted_answer}

Respond with CORRECT or INCORRECT."""
)

eval_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
eval_chain = eval_template | eval_llm | StrOutputParser()

In [26]:
# Evaluate predictions (0.3.26 style)
graded_outputs = []
for i, example in enumerate(examples):
    if i < len(predictions):
        grade = eval_chain.invoke({
            "reference_answer": example.get("answer", ""),
            "predicted_answer": predictions[i]
        })
        graded_outputs.append({"grade": grade})

print(f"Evaluated {len(graded_outputs)} predictions")

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTP

Evaluated 12 predictions


In [43]:
graded_outputs

[{'grade': 'CORRECT'},
 {'grade': 'CORRECT'},
 {'grade': 'INCORRECT'},
 {'grade': 'CORRECT'},
 {'grade': 'INCORRECT'},
 {'grade': 'INCORRECT'},
 {'grade': 'CORRECT'},
 {'grade': 'INCORRECT'},
 {'grade': 'CORRECT'},
 {'grade': 'INCORRECT'},
 {'grade': 'INCORRECT'},
 {'grade': 'CORRECT'}]

In [45]:
# Display results (0.3.26-safe, no duplicate loop)
for i, eg in enumerate(examples):
    question = eg.get("query", eg.get("question", "N/A"))
    real_answer = eg.get("answer", "N/A")
    predicted = predictions[i] if i < len(predictions) else "N/A"
    if not isinstance(predicted, str):
        predicted = predicted.get("result", "N/A")

    print(f"Example {i}:")
    print("Question: " + str(question))
    print("Real Answer: " + str(real_answer))
    print("Predicted Answer: " + str(predicted))

    if i < len(graded_outputs):
        print("Grade: " + str(graded_outputs[i].get("grade", "N/A")))
    else:
        print("Grade: N/A")
    print()

Example 0:
Question: What is the population of New York City as of 2020?
Real Answer: N/A
Predicted Answer: Yes, the Cozy Cuddles Knit Pullover Set has side pockets.
Grade: CORRECT

Example 1:
Question: Which borough of New York City has the highest population?
Real Answer: N/A
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
Grade: CORRECT

Example 2:
Question: What is the economic significance of New York City?
Real Answer: N/A
Predicted Answer: The key features of the Easy-Care Long-Sleeve Gingham Oxford Shirt include:

- 100% cotton, wrinkle-resistant pinpoint fabric
- Relaxed fit that falls at low hip
- Finely tailored with a narrow placket and pearlized buttons
- Taped seams for minimal ironing
- Women's sizes include bust darts
- Machine washable and dryable
- Imported
Grade: INCORRECT

Example 3:
Question: How did New York City get its name?
Real Answer: N/A
Predicted Answer: Small - 18" x 28"; Medium - 22.5" x 34.5".
Grade: CORRE

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [5]:
# Example 2: Text-based QA with Ragas evaluation (0.3.26)
from langchain_huggingface import HuggingFaceEmbeddings

loader = TextLoader('./data/nyc_text.txt')
docs = loader.load()

# Create vectorstore with embeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={'device': 'cpu'})
vector_db_2 = Chroma.from_documents(docs, embedding_model, persist_directory="./chroma_db_nyc")
retriever_2 = vector_db_2.as_retriever(search_kwargs={"k": 2})

# Create QA chain
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

qa_template = ChatPromptTemplate.from_template(
    """Answer the question based on the provided context. Keep the answer concise.

Context:
{context}

Question: {question}
Answer:"""
)

def format_docs_2(docs):
    max_chars_per_doc = 1200
    return "\n\n".join(doc.page_content[:max_chars_per_doc] for doc in docs)

qa_chain = (
    {
        "context": retriever_2 | RunnableLambda(format_docs_2),
        "question": RunnablePassthrough()
    }
    | qa_template
    | llm
    | StrOutputParser()
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10043.78it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
# Test the QA chain (0.3.26 API)
question = "How did New York City get its name?"
result = qa_chain.invoke(question)
print("Question:", question)
print("Answer:", result)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Question: How did New York City get its name?
Answer: The context does not provide information on how New York City got its name.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [ ]:
# In 0.3.26, to get source documents we need a separate retrieval step
source_docs = retriever_2.invoke(question)
print(f"\nSource documents ({len(source_docs)} found):")
for i, doc in enumerate(source_docs[:2]):
    print(f"  Doc {i}: {doc.page_content[:100]}...")


Source documents (2 found):
  Doc 0: New York, often called New York City or NYC, is the most populous city in the United States. With a ...
  Doc 1: New York, often called New York City or NYC, is the most populous city in the United States. With a ...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [20]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [50]:
examples

[{'query': 'What is the population of New York City as of 2020?',
  'ground_truths': ['8,804,190']},
 {'query': 'Which borough of New York City has the highest population?',
  'ground_truths': ['Brooklyn']},
 {'query': 'What is the economic significance of New York City?',
  'ground_truths': ["New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter."]},
 {'query': 'How did New York City

#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain). 
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

In [51]:
# qa_chain expects a plain question string
result = qa_chain.invoke(eval_questions[1])
result

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


'Brooklyn has the highest population among the boroughs of New York City.'

In [52]:
key_mapping = {
    "query": "question",
    "result": "answer",
    "source_documents": "contexts"
}

result_updated = {}
for old_key, new_key in key_mapping.items():
    if old_key in result:
        result_updated[new_key] = result[old_key]


In [53]:
result_updated

{}

In [37]:
# !pip install --no-cache-dir recordclass

In [38]:
# !pip install ragas==0.1.9

In [15]:
# LangChain 0.3.26 compatible Ragas integration (ragas 0.1.9)
try:
    from ragas.integrations.langchain import EvaluatorChain
    from ragas.metrics import (
        faithfulness,
        answer_relevancy,
        context_relevancy,
        context_recall,
    )

    # create evaluation chains
    faithfulness_chain   = EvaluatorChain(metric=faithfulness)
    answer_rel_chain     = EvaluatorChain(metric=answer_relevancy)
    context_rel_chain    = EvaluatorChain(metric=context_relevancy)
    context_recall_chain = EvaluatorChain(metric=context_recall)
    print("✓ Ragas evaluators initialized successfully")
except ImportError as e:
    print(f"⚠️  Ragas import note: {str(e)[:60]}")

✓ Ragas evaluators initialized successfully


1. `__call__()`

Directly run the evaluation chain with the results from the QA chain. Do note that metrics like context_relevancy and faithfulness require the `source_documents` to be present.

In [10]:
# Recheck the result that we are going to validate.
result

'The context does not provide information on how New York City got its name.'

**Faithfulness**

In [21]:
# Reformat result for Ragas (0.3.26 + ragas 0.1.9)
source_docs = retriever_2.invoke(question)
contexts = [doc.page_content[:1200] for doc in source_docs[:2]]
eval_input = {
    "question": question,
    "answer": result,
    "contexts": contexts
}

# Evaluate with Ragas (async-safe in notebooks)
eval_result = await faithfulness_chain.acall(eval_input)
print(f"Faithfulness Score: {eval_result.get('faithfulness', 'N/A')}")

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Faithfulness Score: 0.0


High faithfulness_score means that there are exact consistency between the source documents and the answer.

You can check lower faithfulness scores by changing the result (answer from LLM) or source_documents to something else.

In [23]:
# Create a deliberately wrong answer and re-evaluate faithfulness
contexts = [doc.page_content[:1200] for doc in source_docs[:2]]
fake_eval_input = {
    "question": question,
    "answer": "we are the champions",
    "contexts": contexts
}

eval_result = await faithfulness_chain.acall(fake_eval_input)
eval_result.get("faithfulness", "N/A")

No statements were generated from the answer.


nan

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


**Context Relevancy**

In [24]:
# Evaluate context recall with ground truth
ground_truth = eval_answers[1]
contexts = [doc.page_content[:1200] for doc in source_docs[:2]]

eval_input_recall = {
    "question": eval_questions[1],
    "answer": result,
    "contexts": contexts,
    "ground_truth": ground_truth
}

eval_result = await context_recall_chain.acall(eval_input_recall)
print(f"Context Recall Score: {eval_result.get('context_recall', 'N/A')}")

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Context Recall Score: 0.0


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


High context_recall_score means that the ground truth is present in the source documents.

You can check lower context recall scores by changing the source_documents to something else.

In [25]:
from langchain.schema import Document

fake_docs = [Document(page_content="I love christmas")]
fake_eval_input = {
    "question": eval_questions[1],
    "answer": result,
    "contexts": [doc.page_content[:1200] for doc in fake_docs],
    "ground_truth": eval_answers[1]
}

eval_result = await context_recall_chain.acall(fake_eval_input)
eval_result.get("context_recall", "N/A")

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


0.0

2. `evaluate()`

Evaluate a list of inputs/queries and the outputs/predictions from the QA chain.

In [26]:
# Batch evaluation with Ragas (0.3.26 + ragas 0.1.9)
print("Running batch evaluation (this may take a minute)...")

eval_results = []
for i, question in enumerate(eval_questions):
    try:
        answer = qa_chain.invoke(question)
        docs = retriever_2.invoke(question)

        eval_input = {
            "question": question,
            "answer": answer,
            "contexts": [doc.page_content[:1200] for doc in docs[:2]],
            "ground_truth": eval_answers[i]
        }

        f_result = await faithfulness_chain.acall(eval_input)
        score = f_result.get("faithfulness", 0)
        eval_results.append({
            "question": question,
            "faithfulness": score
        })
        print(f"  [{i+1}/{len(eval_questions)}] {question[:40]}... → Faithfulness: {score:.2f}")
    except Exception as e:
        print(f"  [{i+1}/{len(eval_questions)}] Error: {str(e)[:60]}")

print(f"\nEvaluated {len(eval_results)} queries")
eval_results

Running batch evaluation (this may take a minute)...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  [1/5] What is the population of New York City ... → Faithfulness: 1.00


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  [2/5] Which borough of New York City has the h... → Faithfulness: 0.00


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


... (2 outputs left out)

  [4/5] How did New York City get its name?... → Faithfulness: 0.00


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


  [5/5] What is the significance of the Statue o... → Faithfulness: 0.00

Evaluated 5 queries


[{'question': 'What is the population of New York City as of 2020?',
  'faithfulness': 1.0},
 {'question': 'Which borough of New York City has the highest population?',
  'faithfulness': 0.0},
 {'question': 'What is the economic significance of New York City?',
  'faithfulness': 1.0},
 {'question': 'How did New York City get its name?', 'faithfulness': 0.0},
 {'question': 'What is the significance of the Statue of Liberty in New York City?',
  'faithfulness': 0.0}]

In [27]:
# Evaluate context recall for all questions (0.3.26 + ragas 0.1.9)
print("Evaluating context recall...")

recall_results = []
for i, question in enumerate(eval_questions):
    try:
        answer = qa_chain.invoke(question)
        docs = retriever_2.invoke(question)

        eval_input = {
            "question": question,
            "answer": answer,
            "contexts": [doc.page_content[:1200] for doc in docs[:2]],
            "ground_truth": eval_answers[i]
        }

        r_result = await context_recall_chain.acall(eval_input)
        recall_results.append({
            "question": question[:30],
            "context_recall": r_result.get("context_recall", 0)
        })
    except Exception as e:
        print(f"Error evaluating question {i}: {str(e)[:60]}")

print("Context Recall Results:")
for r in recall_results:
    print(f"  {r['question']}... → {r['context_recall']:.2f}")

Evaluating context recall...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTP

Context Recall Results:
  What is the population of New ... → 1.00
  Which borough of New York City... → 0.00
  What is the economic significa... → 0.20
  How did New York City get its ... → 1.00
  What is the significance of th... → 0.00
